In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import uproot
import sys
import math
import os
import ROOT

dir = "/Users/alexanderantonakis/Desktop/Software/AFrameAnalysis/Macros/"

sys.path.append("../Utils")
sys.path.append("../Configs")
sys.path.append("../DAQ")
sys.path.append("../Mappings")
sys.path.append("../BiasFiles")
sys.path.append("../SiPM_HISTS")

from frame_to_crt_fcl import *



NEW_FCL_DIR = "../DAQ/FEB_FCL_Conservative_95_SiPM/"


daq_df = pd.DataFrame(frame_map, columns = ["Frame", "Frame_FEB", "DAQ_FEB", "Wall"])


# Put the directory that you want to SiPM correct
#strip_fcl_dir = "../DAQ/FEB_FCL_95/"

strip_fcl_dir = "../DAQ/FEB_FCL_Conservative_95/"

all_fcl_files = [f for f in os.listdir(strip_fcl_dir) if os.path.isfile(strip_fcl_dir+f)]
all_sipm_hists = [f for f in os.listdir("../SiPM_HISTS/") if os.path.isfile("../SiPM_HISTS/"+f)]


print("All SiPM Hists available")
print(all_sipm_hists)

daq_df[:4]

In [ ]:
from run_to_voltage import*
volt_df = pd.DataFrame(run_to_volt, columns = ["Frame", "Run", "V"])
volt_df[:4]

In [ ]:
daq_col = list(daq_df['DAQ_FEB'].values)
print(daq_col)

In [ ]:
def get_settings(fcl):
    settings = []
    count = 0
    with open(fcl, 'r') as file:
        # Loop through each line in the file
        for line in file:
        # Strip the newline character and split the line by whitespace
            temp_line = line.strip().split()
            if len(temp_line) > 0:
                if temp_line[0] == "[":
                    #temp_line = temp_line.split(",")
                    #print(temp_line[4])
                    val = int(temp_line[4].split(",")[0])
                    settings.append(val)
    return settings

In [ ]:
def make_new_fcl(name, settings, daq_feb):
    new_lines = []
    std_fcl = "../DAQ/east_rates_nominal/feb018.fcl"
    count = 0
    with open(std_fcl, 'r') as file:
        # Loop through each line in the file
        for line in file:
            if line == "FEB018Configuration:  @local::FEBConfigurationStandard"+"\n":
                new_lines.append("FEB"+daq_feb+"Configuration:  @local::FEBConfigurationStandard"+"\n")
                continue
                #print("WHOA")
            if line == "FEB018Configuration.channel_configuration: ["+"\n":
                new_lines.append("FEB"+daq_feb+"Configuration"+".channel_configuration: ["+"\n")
                continue
          
        # Strip the newline character and split the line by whitespace
            temp_line = line.strip().split()
            if len(temp_line) > 0:
                if temp_line[0] == "[":
                    #print(temp_line)
                    i = temp_line.index('170,')
                    new_line = temp_line
                    if settings[count] < 140:
                        settings[count] = 140
                    new_line[i] = str(settings[count]) + ","
                    count += 1               
                    final_line = ""
                    for w in new_line:
                        final_line += w
                        final_line += " "
                    final_line += "\n"
                    new_lines.append(final_line)
                else:
                    new_lines.append(line)
            else:
                new_lines.append(line)
    
    # Open new file in write mode
    with open(NEW_FCL_DIR+name, 'w') as file:
        # Loop through each line in the list
        for line in new_lines:
            # Write the line to the file
            file.write(line)

In [ ]:
# loop over the fcl files and make new ones ...
for fcl in all_fcl_files:
    daq_feb = fcl.split(".fcl")[0].split("feb")[1]
    print("DAQ FEB", daq_feb )
    i = daq_col.index(daq_feb)
    frame_feb = daq_df.iloc[i]['Frame_FEB']

    # Add exception for FEB 137 becasue it saw nothing :(
    if frame_feb == 137:
        print("Skipping FEB 137 !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        os.system("cp "+strip_fcl_dir+fcl+" "+NEW_FCL_DIR+fcl)
        continue
        
    frame = daq_df.iloc[i]['Frame']
    print("Frame FEB", frame_feb)

    runs = volt_df.query("Frame == "+str(frame))["Run"].values
    voltages = volt_df.query("Frame == "+str(frame))["V"].values

    sorted_voltages, sorted_runs = zip(*sorted(zip(voltages, runs)))

    # Convert them back to lists (zip returns tuples)
    sorted_voltages = list(sorted_voltages)
    sorted_runs = list(sorted_runs)
    print("Runs", sorted_runs)
    print("volt", sorted_voltages)
    
    fractions = [[] for num in range(len(sorted_runs))]
    f = ROOT.TFile.Open("../SiPM_HISTS/sipm_hists_frame"+str(frame)+".root", "READ")
    count = 0
    for run in runs:
        odd_name = str(run)+"_"+str(frame_feb)+"_odd"
        even_name = str(run)+"_"+str(frame_feb)+"_even"
        h_odd = f.Get(odd_name)
        h_even = f.Get(even_name)
        h_ratio = h_even.Clone("h_ratio_"+str(frame_feb))
        h_ratio.Divide(h_odd)
        for num in range(16):
            fractions[count].append(h_ratio.GetBinContent(num+1))
            
        count += 1
    
    print("")
    s = get_settings(strip_fcl_dir+fcl)
    print(s)

    new_settings = []

    for num in range(16):
        a = num*2 # even 
        b = num*2+1 # odd
        #num += 2
        if s[a] != s[b]:
            print("We have a problem !!!! Strip Cahnnels don't match !!!! ;(")
            break

        if s[a] < min(sorted_voltages):
            # interpolate below
            p1 = [sorted_voltages[-2], fractions[-2][num]] 
            p2 = [sorted_voltages[-1], fractions[-1][num]]
            dx = p2[0] - p1[0]
            dy = p2[1] - p1[1]
            m = dy/dx
            b = p2[1] - m*p2[0]
            corr = m*s[a] + b
            if corr < 1:
                # even is smaller --> correct the odd
                new_settings.append(s[a])
                new_settings.append(corr*s[a])
            else:
                # even is larger --> correct the even
                new_settings.append((1.0/corr)*s[a])
                new_settings.append(s[a])
        
        elif s[a] > max(sorted_voltages):
            # interpolate above
            p1 = [sorted_voltages[1], fractions[1][num]] 
            p2 = [sorted_voltages[0], fractions[0][num]]
            dx = p2[0] - p1[0]
            dy = p2[1] - p1[1]
            m = dy/dx
            b = p2[1] - m*p2[0]
            corr = m*s[a] + b
            if corr < 1:
                # even is smaller --> correct the odd
                new_settings.append(s[a])
                new_settings.append(corr*s[a])
            else:
                # even is larger --> correct the even
                new_settings.append((1.0/corr)*s[a])
                new_settings.append(s[a])
            
        else:
            # We are somewhere in bewtween
            stop_idx = -1
            for j in range(len(sorted_voltages)):
                if sorted_voltages[j] > s[a]:
                    stop_idx = j
                    break
                    
            p1 = [sorted_voltages[j-1], fractions[j-1][num]] 
            p2 = [sorted_voltages[j], fractions[j][num]]
            dx = p2[0] - p1[0]
            dy = p2[1] - p1[1]
            m = dy/dx
            b = p2[1] - m*p2[0]
            corr = m*s[a] + b
            if corr < 1:
                # even is smaller --> correct the odd
                new_settings.append(s[a])
                new_settings.append(corr*s[a])
            else:
                # even is larger --> correct the even
                new_settings.append((1.0/corr)*s[a])
                new_settings.append(s[a])

    for num in range(len(new_settings)):
        new_settings[num] = math.ceil(new_settings[num])
    
    print(new_settings)
    make_new_fcl(fcl, new_settings, daq_feb)
    print("")
    
print("Finished !!!")   